In [1]:
import optuna
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

DATA_DIR = Path.cwd().parent / 'data' / 'processed'
RANDOM_STATE = 25

X_train_full = pd.read_parquet(DATA_DIR / 'X_train.parquet')
y_train_full = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']
y_int, _ = pd.factorize(y_train_full, sort=True)
y_int = pd.Series(y_int, index=y_train_full.index)

# Split into train and validation for this simple version
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_full, y_int, test_size=0.2, stratify=y_int, random_state=RANDOM_STATE
)

numeric_cols     = X_train_full.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train_full.select_dtypes(include=['category', 'bool']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])

def objective(trial):
    n_estimators  = trial.suggest_int('n_estimators', 100, 500)
    max_depth     = trial.suggest_int('max_depth', 3, 8)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)

    pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
            tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
        )),
    ])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_va)
    return f1_score(y_va, y_pred, average='macro')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print(f"Best macro-F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-07-28 14:48:43,978] A new study created in memory with name: no-name-50c0df26-eaed-4516-b151-dcf8aa7cb7a5
[I 2026-07-28 14:48:56,257] Trial 0 finished with value: 0.4102308711925595 and parameters: {'n_estimators': 444, 'max_depth': 5, 'learning_rate': 0.07628134274817908}. Best is trial 0 with value: 0.4102308711925595.
[I 2026-07-28 14:49:04,022] Trial 1 finished with value: 0.3775596224571118 and parameters: {'n_estimators': 227, 'max_depth': 4, 'learning_rate': 0.05583465527816895}. Best is trial 0 with value: 0.4102308711925595.
[I 2026-07-28 14:49:14,901] Trial 2 finished with value: 0.3943100224513249 and parameters: {'n_estimators': 328, 'max_depth': 5, 'learning_rate': 0.054384831806659734}. Best is trial 0 with value: 0.4102308711925595.
[I 2026-07-28 14:49:23,400] Trial 3 finished with value: 0.40398453767376735 and parameters: {'n_estimators': 239, 'max_depth': 6, 'learning_rate': 0.08600620359127759}. Best is trial 0 with value: 0.4102308711925595.
[I 2026-07-28 14

Best macro-F1: 0.4290
Best params: {'n_estimators': 320, 'max_depth': 7, 'learning_rate': 0.14187649598252935}
